In [0]:
%run ../00_common/calc_ctable

In [0]:
%run ../00_common/UDF_utils

In [0]:
# 启用membership mapping 
MARKETS_ENABLE_MEM_MAPPING = ["KOR"]

ts_format = "yyyy-MM-dd'T'HH:mm:ss"

In [0]:
def get_membership_mapping_data(task_id):
    """
    业务逻辑：
      Part1 - 正常记录：t_clean_program JOIN t_clean_consumer，
              过滤 prgt_code 在 t_membership_program_code 中的记录
      Part2 - DELETE 记录（program 在 master 有但 source 中已不存在）：
              source 中有该 consumer，但对应的 program code 在 source 中已消失
      Part3 - DELETE 记录（consumer 在 source 中已是 DELETE 状态）：
              source consumer 动作为 DELETE，且 master 中有对应 program 记录
    """

    # ----------------------------------------------------------
    # 1. 读取所有依赖表
    # ----------------------------------------------------------
    clean_consumer_df  = (spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer")
        .filter(F.col("task_id") == task_id)
        .filter(F.col("is_include") == True)
        .filter(F.col("srcc_mrkt_code").isin(MARKETS_ENABLE_MEM_MAPPING))
    )

    clean_program_df   = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_program")

    membership_code_df = spark.table(f"{get_env_config('config_database')}.t_membership_program_code")

    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_program_df  = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_program")

    # ----------------------------------------------------------
    # 2. 公共子集：有效的 (prgt_code, MarketCode) 组合
    # ----------------------------------------------------------
    valid_prgt_df = (
        membership_code_df
        .select(
            F.col("prgt_code"),
            F.col("MarketCode").alias("prgt_mrkt_code"),  # 新增 MarketCode 维度
        )
        .distinct()
    )

    part1_df = (
        clean_program_df.alias("cp")
        .join(
            clean_consumer_df.alias("cc"),
            (F.col("cp.SRPG_SRCC_ID") == F.col("cc.SRCC_ID"))
            & (F.col("cp.SRPG_MRKT_CODE") == F.col("cc.SRCC_MRKT_CODE")),  # 新增
            "inner",
        )
        .join(
            valid_prgt_df.alias("vpc"),
            (F.col("cp.SRPG_PRGT_CODE") == F.col("vpc.prgt_code"))
            & (F.col("cp.SRPG_MRKT_CODE") == F.col("vpc.prgt_mrkt_code")),  # 新增
            "inner",
        )
        .select(
            F.col("cc.SRCC_ID"),
            F.col("cc.SRCC_ACTION"),
            F.col("cc.SRCC_SOURCETIMESTAMP"),
            F.col("cp.SRPG_PRGT_CODE"),
            F.col("cc.SRCC_MRKT_CODE"),
            F.col("cc.SRCC_BRND_CODE"),
            F.col("cp.SRPG_MEMBERSHIPNUM"),
            F.col("cc.SRCC_SRCS_CODE"),
            F.col("cc.SRCC_CONSUMERID"),
        )
        .distinct()
    )


    # source 中当前仍存在的有效 (prgt_code, mrkt_code) 组合
    existing_source_prgt = (
        clean_program_df.alias("cp2")
        .join(
            valid_prgt_df.alias("vpc2"),
            (F.col("cp2.SRPG_PRGT_CODE") == F.col("vpc2.prgt_code"))
            & (F.col("cp2.SRPG_MRKT_CODE") == F.col("vpc2.prgt_mrkt_code")),  # 新增
            "inner",
        )
        .select(
            F.col("cp2.SRPG_PRGT_CODE").alias("exist_prgt_code"),
            F.col("cp2.SRPG_MRKT_CODE").alias("exist_mrkt_code"),  # 新增
        )
        .distinct()
    )

    part2_df = (
        clean_program_df.alias("sp")
        .join(
            clean_consumer_df.alias("sc"),
            (F.col("sp.SRPG_SRCC_ID") == F.col("sc.SRCC_ID"))
            & (F.col("sp.SRPG_MRKT_CODE") == F.col("sc.SRCC_MRKT_CODE")),  # 新增
            "inner",
        )
        .join(
            master_consumer_df.alias("mc"),
            (F.col("sc.SRCC_SRCS_CODE") == F.col("mc.SCON_SRCS_CODE"))
            & (F.col("sc.SRCC_MRKT_CODE") == F.col("mc.SCON_MRKT_CODE"))
            & (F.col("sc.SRCC_BRND_CODE") == F.col("mc.SCON_BRND_CODE"))
            & (F.col("sc.SRCC_CONSUMERID") == F.col("mc.SCON_CONSUMERID")),
            "inner",
        )
        .join(
            master_program_df.alias("mp"),
            (F.col("mc.SCON_ID") == F.col("mp.SCPR_SCON_ID"))
            & (F.col("mc.SCON_MRKT_CODE") == F.col("mp.SCPR_MRKT_CODE")),  # 新增
            "left",
        )
        .join(
            valid_prgt_df.alias("vpc3"),
            (F.col("mp.SCPR_PRGT_CODE") == F.col("vpc3.prgt_code"))
            & (F.col("mp.SCPR_MRKT_CODE") == F.col("vpc3.prgt_mrkt_code")),  # 新增
            "inner",
        )
        # 排除 source 中仍存在的 (prgt_code, mrkt_code) 组合
        .join(
            existing_source_prgt,
            (F.col("mp.SCPR_PRGT_CODE") == F.col("exist_prgt_code"))
            & (F.col("mp.SCPR_MRKT_CODE") == F.col("exist_mrkt_code")),  # 新增
            "left_anti",
        )
        .filter(
            (F.col("mp.SCPR_MEMBERSHIPNUM").isNotNull())
            & (F.col("mp.SCPR_MEMBERSHIPNUM") != "")
        )
        .select(
            F.col("sc.SRCC_ID"),
            F.lit("DELETE").alias("SRCC_ACTION"),
            F.col("sc.SRCC_SOURCETIMESTAMP"),
            F.col("mp.SCPR_PRGT_CODE").alias("SRPG_PRGT_CODE"),
            F.col("sc.SRCC_MRKT_CODE"),
            F.col("sc.SRCC_BRND_CODE"),
            F.col("mp.SCPR_MEMBERSHIPNUM").alias("SRPG_MEMBERSHIPNUM"),
            F.col("sc.SRCC_SRCS_CODE"),
            F.col("sc.SRCC_CONSUMERID"),
        )
        .distinct()
    )

    part3_df = (
        clean_consumer_df.alias("sc3")
        .join(
            clean_program_df.alias("sp3"),
            (F.col("sp3.SRPG_SRCC_ID") == F.col("sc3.SRCC_ID"))
            & (F.col("sp3.SRPG_MRKT_CODE") == F.col("sc3.SRCC_MRKT_CODE")),  # 新增
            "left",
        )
        .join(
            master_consumer_df.alias("mc3"),
            (F.col("sc3.SRCC_SRCS_CODE") == F.col("mc3.SCON_SRCS_CODE"))
            & (F.col("sc3.SRCC_MRKT_CODE") == F.col("mc3.SCON_MRKT_CODE"))
            & (F.col("sc3.SRCC_BRND_CODE") == F.col("mc3.SCON_BRND_CODE"))
            & (F.col("sc3.SRCC_CONSUMERID") == F.col("mc3.SCON_CONSUMERID")),
            "inner",
        )
        .join(
            master_program_df.alias("mp3"),
            (F.col("mc3.SCON_ID") == F.col("mp3.SCPR_SCON_ID"))
            & (F.col("mc3.SCON_MRKT_CODE") == F.col("mp3.SCPR_MRKT_CODE")),  # 新增
            "inner",
        )
        .join(
            valid_prgt_df.alias("vpc4"),
            (F.col("mp3.SCPR_PRGT_CODE") == F.col("vpc4.prgt_code"))
            & (F.col("mp3.SCPR_MRKT_CODE") == F.col("vpc4.prgt_mrkt_code")),  # 新增
            "inner",
        )
        .filter(
            (F.col("sc3.SRCC_ACTION") == "DELETE")
            & (F.col("mp3.SCPR_MEMBERSHIPNUM").isNotNull())
            & (F.col("mp3.SCPR_MEMBERSHIPNUM") != "")
        )
        .select(
            F.col("sc3.SRCC_ID"),
            F.lit("DELETE").alias("SRCC_ACTION"),
            F.col("sc3.SRCC_SOURCETIMESTAMP"),
            F.col("mp3.SCPR_PRGT_CODE").alias("SRPG_PRGT_CODE"),
            F.col("sc3.SRCC_MRKT_CODE"),
            F.col("sc3.SRCC_BRND_CODE"),
            F.col("mp3.SCPR_MEMBERSHIPNUM").alias("SRPG_MEMBERSHIPNUM"),
            F.col("sc3.SRCC_SRCS_CODE"),
            F.col("sc3.SRCC_CONSUMERID"),
        )
        .distinct()
    )

    # ----------------------------------------------------------
    # 3. UNION 三部分，去重，按时间排序
    # ----------------------------------------------------------
    result_df = (
        part1_df
        .unionByName(part2_df)
        .unionByName(part3_df)
        .distinct()
        .orderBy(F.col("SRCC_SOURCETIMESTAMP"))
    )

    return result_df


In [0]:
def calc_t_membership_mapping_log(task_id):

    membership_mapping_df = get_membership_mapping_data(task_id)

    # generate json string
    result_df = (membership_mapping_df
        .withColumn("document_uuid", F.expr("uuid()"))
        .withColumn("record_uuid", F.expr("uuid()"))
        .withColumn("DocumentTimestamp",F.date_format(F.current_timestamp(), ts_format))
        .withColumn("MappingTimestamp",F.date_format(F.col("SRCC_SOURCETIMESTAMP"), ts_format))
        .withColumn("json_str",
            build_cid_mapping_json_udf(
                F.col("SRCC_ACTION"),
                F.col("SRPG_PRGT_CODE"),
                F.col("SRCC_SRCS_CODE"),
                F.col("SRCC_MRKT_CODE"),
                F.col("SRCC_BRND_CODE"),
                F.col("SRPG_MEMBERSHIPNUM"),
                F.col("SRCC_CONSUMERID"),
                F.col("DocumentTimestamp"),             # DocumentTimestamp
                F.col("document_uuid"),                 # DocumentUUID
                F.col("record_uuid"),                   # @RecordUUID
                F.col("MappingTimestamp"),              # MappingTimestamp
            )
        )
        .select(
            F.col("record_uuid"),
            F.col("SRCC_ID"),
            F.col("cc.SRCC_MRKT_CODE"),
            F.col("cc.SRCC_BRND_CODE"),
            F.col("cc.SRCC_SRCS_CODE"),
            F.col("cc.SRCC_CONSUMERID"),
            F.col("cp.SRPG_PRGT_CODE"),
            F.col("cp.SRPG_MEMBERSHIPNUM"),
            F.col("SRCC_SOURCETIMESTAMP"),
            F.col("json_str"),
            F.current_timestamp().alias("creation_dt"),
            F.lit(task_id).alias("task_id")
        )
    )

    result_df.cache()
    result_count = result_df.count()
    print(f"Total records to insert: {result_count}")

    if result_count > 0:
        target_table_name = f"{get_env_config('golden_consumer_combine_database')}.t_membership_mapping_log"
        append_table(result_df, target_table_name)

        # calc ctable
        calc_ctable(target_table_name, None)

    # unpersist
    result_df.unpersist()

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")


with StepLogger("3.4_generate_membership_mapping", "03-4", "consumerlist", task_id=task_id) as logger:
    calc_t_membership_mapping_log(task_id)